# Pacman Training Notebook

In [ ]:

import gymnasium as gym
from gymnasium.wrappers import AtariPreprocessing, FrameStack
import numpy as np
import torch, torch.nn as nn, torch.optim as optim
import matplotlib.pyplot as plt

env = FrameStack(
    AtariPreprocessing(gym.make("ALE/MsPacman-v5"), grayscale_obs=True),
    num_stack=4
)

class CNN(nn.Module):
    def __init__(self):
        super().__init__()
        self.conv = nn.Sequential(
            nn.Conv2d(4, 32, 8, stride=4), nn.ReLU(),
            nn.Conv2d(32, 64, 4, stride=2), nn.ReLU(),
            nn.Conv2d(64, 64, 3, stride=1), nn.ReLU()
        )
        self.fc = nn.Sequential(
            nn.Linear(64*7*7, 512), nn.ReLU(),
            nn.Linear(512, env.action_space.n)
        )

    def forward(self, x):
        x = x / 255.0
        x = self.conv(x)
        x = x.view(x.size(0), -1)
        return self.fc(x)

net = CNN()
opt = optim.Adam(net.parameters(), lr=1e-4)
loss_fn = nn.MSELoss()
gamma = 0.99
rewards = []

for episode in range(20):
    s,_ = env.reset()
    s = np.array(s)
    total = 0
    for t in range(5000):
        s_t = torch.tensor(s, dtype=torch.float32).unsqueeze(0)
        q = net(s_t)
        a = torch.argmax(q).item()

        s2, r, done, trunc, _ = env.step(a)
        s2 = np.array(s2)
        total += r

        with torch.no_grad():
            target = q.clone()
            q2 = net(torch.tensor(s2, dtype=torch.float32).unsqueeze(0))
            target[0, a] = r + gamma * torch.max(q2).item() * (not done)

        loss = loss_fn(q, target)
        opt.zero_grad(); loss.backward(); opt.step()

        s = s2
        if done: break
    rewards.append(total)

plt.plot(rewards); plt.title("Pacman Training Reward"); plt.show()

print("Training complete.")
